# HW4P1: Decoder-Only Language Modeling

In this assignment, you will implement and train a decoder-only Transformer.
Complete each implementation section and run the test directly below it before
starting your experiments.


# Setup

Follow the setup instructions for the environment you are using. Keep this notebook
in the handout repository, and do not save API keys or access tokens in it.


## Local

We recommend completing the implementation tests locally before moving to a GPU.

### Create and activate the environment

```bash
conda create -n hw4 python=3.12.4
conda activate hw4
pip install --no-cache-dir -r requirements.txt
```

Open this notebook from the handout directory and select the `hw4` environment as
the kernel. The repository should contain at least:

```text
hw4lib/
mytorch/
tests/
requirements.txt
```

Local CPU is sufficient for the implementation tests. Use a supported GPU
environment for training.


## Colab

### Step 1: Get the handout

Upload or clone your handout repository using the method described in the writeup.
Do not place a private GitHub token in a clone URL saved in this notebook.

### Step 2: Install dependencies

Change the path below if your repository has a different name.
Use Colab's installed PyTorch/CUDA stack. The Colab requirements add the notebook
packages without the local environment's PyTorch pin.


In [ ]:
%cd /content/IDL-26Fall-HW4
%pip install -q -r requirements-colab.txt

import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)


### Step 3: Get the data

Open the course's private data-sharing link in your browser. Download version 1
and extract it into Colab storage or Google Drive. Set `manual_data_root` below
to the folder containing `mini_train`, `mini_val`, `train`, `val`, and `test`.
For Drive, use `/content/drive/MyDrive/...`; the cell mounts Drive when needed.

Leave the path empty to try KaggleHub instead. Browser link access does not
guarantee API access. If the import or download fails (including 403), use the
manual path and rerun the cell; it skips KaggleHub installation and login.
Do not save credentials in the notebook. Drive reads may be slower than local
storage; you can copy the extracted folder to `/content` before training.


In [ ]:
import os
from pathlib import Path

# For a browser download, extract the course dataset first and set this path.
# Example: "/content/drive/MyDrive/hw4p1-data" or "/content/hw4p1-data"
manual_data_root = os.environ.get("HW4P1_DATA_ROOT", "")

if manual_data_root:
    if manual_data_root.startswith("/content/drive/") and not Path(manual_data_root).exists():
        from google.colab import drive
        drive.mount("/content/drive")
    data_root = str(Path(manual_data_root).expanduser().resolve())
else:
    # API download is optional. A browser sharing link may work while this returns 403.
    import subprocess
    import sys
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
        kagglehub.login()
        data_root = kagglehub.dataset_download("cmu11785/hw4p1-data/versions/1")
    except Exception as exc:
        raise RuntimeError(
            "KaggleHub setup/download failed. Download version 1 through the course "
            "sharing link in your browser, extract it into Colab or Google Drive, "
            "set manual_data_root above, and rerun this cell. "
            "Use the folder containing mini_train, mini_val, train, val, and test."
        ) from exc

for partition in ("mini_train", "mini_val", "train", "val", "test"):
    if not (Path(data_root) / partition / "manifest.json").is_file():
        raise FileNotFoundError(
            f"Missing {partition}/manifest.json under {data_root}. "
            "Set manual_data_root to the extracted dataset folder, not the ZIP or its parent."
        )
os.environ["HW4P1_DATA_ROOT"] = data_root
print("Data root:", data_root)


### Step 4: Move to the handout directory

You may need to repeat this after restarting the runtime.


In [ ]:
%cd /content/IDL-26Fall-HW4
!pwd
!ls


## Kaggle

### Step 1: Get the handout

Open the course notebook and choose **Copy & Edit**, or upload the handout archive
from the course site. Keep editable files under `/kaggle/working`.

### Step 2: Select a GPU

Open Notebook Settings and select a GPU accelerator.

### Step 3: Add the data

Sign in to the account you will use for the assignment and open the private HW4P1
data-sharing link from the course site. Add `hw4p1-data` through the notebook Input
panel. Do not repost the sharing link.

### Step 4: Move to the handout directory

Change the path below if your handout directory has a different name.


In [ ]:
%cd /kaggle/working/IDL-26Fall-HW4
!pwd
!ls


In [ ]:
import os
from pathlib import Path

# If you attached the data with Add Input, set its /kaggle/input/... folder here.
manual_data_root = os.environ.get("HW4P1_DATA_ROOT", "")
if manual_data_root:
    data_root = str(Path(manual_data_root).expanduser().resolve())
else:
    try:
        import kagglehub
        data_root = kagglehub.dataset_download("cmu11785/hw4p1-data/versions/1")
    except Exception as exc:
        raise RuntimeError(
            "Kaggle download failed. Attach the course data using Add Input, "
            "set manual_data_root to its extracted dataset folder, and rerun."
        ) from exc
for partition in ("mini_train", "mini_val", "train", "val", "test"):
    if not (Path(data_root) / partition / "manifest.json").is_file():
        raise FileNotFoundError(f"Missing {partition}/manifest.json under {data_root}")
os.environ["HW4P1_DATA_ROOT"] = data_root
print("Data root:", data_root)


## PSC

### Step 1: Work from your Jet home directory

Keep the handout, notebooks, checkpoints, and experiment outputs under your Jet home
directory. Do not write checkpoints to Ocean storage.

```bash
cd /jet/home/$USER
```

### Step 2: Request a GPU and activate the course environment

Use the course allocation, GPU partition, and conda environment announced on the
course site.

```bash
interact -p <COURSE_GPU_PARTITION> --gres=gpu:<COURSE_GPU_TYPE>:1 \
  -t <COURSE_TIME_LIMIT> -A <COURSE_ALLOCATION>
module load anaconda3
conda activate <COURSE_CONDA_ENV>
export PYTHONNOUSERSITE=1
```

### Step 3: Set data and output paths

The course data is shared read-only from Ocean. Do not copy checkpoints or other
experiment outputs into this directory; keep them in Jet.

```bash
export HW4P1_DATA_ROOT=/ocean/projects/cis240101p/pchang4/hw4p1_data
export HW4P1_OUTPUT_ROOT=/jet/home/$USER/hw4p1_runs
mkdir -p "$HW4P1_OUTPUT_ROOT"
```

### Step 4: Start Jupyter

```bash
cd /jet/home/$USER/IDL-26Fall-HW4
jupyter notebook --no-browser --ip=0.0.0.0
```

Connect using the Jupyter URL printed in the terminal and select the course kernel.


# Imports

Run this notebook from the `IDL-26Fall-HW4/` folder containing `hw4lib/`.
If you extracted the handout ZIP, use the extracted folder.


In [ ]:
from pathlib import Path
import gc
import json
import os
import random
import shutil
import sys

import numpy as np
import torch
import yaml
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "hw4lib").is_dir():
    PROJECT_ROOT = PROJECT_ROOT / "IDL-26Fall-HW4"
assert (PROJECT_ROOT / "hw4lib").is_dir(), "Run the notebook from its folder."
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hw4lib.data import GPT2Tokenizer, LMDataset, verify_dataloader
from hw4lib.data.window_sampler import EpochWindowSampler
from hw4lib.model import DecoderOnlyTransformer
from hw4lib.trainers import LMTrainer
from hw4lib.utils import create_optimizer, create_scheduler

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Project root:", PROJECT_ROOT)
print("Using device:", device)


# Implementations

Read the specifications in each source file before filling in the TODOs.
The tests use small CPU models; their parameter counts are not your experiment model.


## MyTorch Components from HW3P1

Copy your own completed `Softmax`, `Linear`, `ScaledDotProductAttention`, and
`MultiHeadAttention` implementations from HW3P1 into `mytorch/nn`. These
components are required prerequisite checks but carry no HW4P1 points. Run each check before starting GQA.

For attention, a fully masked query row must have zero probabilities and zero
output, with no NaNs or gradient contribution from that row. Check this when
copying your HW3 implementation; subtracting a constant from the entire row is
not enough.


In [ ]:
!{sys.executable} -m tests.test_mytorch_softmax


In [ ]:
!{sys.executable} -m tests.test_mytorch_linear


In [ ]:
!{sys.executable} -m tests.test_mytorch_scaled_dot_product_attention


In [ ]:
!{sys.executable} -m tests.test_mytorch_multi_head_attention


## Grouped-Query and Multi-Query Attention

Complete `GroupedQueryAttention` in `mytorch/nn`. `MultiQueryAttention` is the
one-KV-head case and inherits your GQA implementation.


In [ ]:
!{sys.executable} -m tests.test_mytorch_grouped_query_attention


## GPT-2 Tokenizer and LM Dataset

The GPT-2 tokenizer and shard format are provided. Complete the fixed-window LM
dataset in `hw4lib/data/lm_dataset.py`.


In [ ]:
!{sys.executable} -m tests.test_gpt2_data


## Positional Encoding

Complete `PositionalEncoding` and `RotaryPositionalEncoding` in
`hw4lib/model/positional_encoding.py`.


In [ ]:
!{sys.executable} -m tests.test_positional_encoding


In [ ]:
!{sys.executable} -m tests.test_rotary_positional_encoding


## Causal Self-Attention

Use your RoPE implementation to complete causal MHA/GQA/MQA in
`hw4lib/model/causal_self_attention.py`.


In [ ]:
!{sys.executable} -m tests.test_causal_self_attention


## KV Cache

Complete the compact KV-cache path in `CausalSelfAttention`. Store only the
KV heads and use the cached length as the RoPE position offset.


In [ ]:
!{sys.executable} -m tests.test_kv_cache


## Masks

Complete `PadMask` and `CausalMask` in `hw4lib/model/masks.py`. Both are graded.


In [ ]:
!{sys.executable} -m tests.test_mask_padding


In [ ]:
!{sys.executable} -m tests.test_mask_causal


## Self-Attention Layer

Complete `SelfAttentionLayer`, including its cached path.


In [ ]:
!{sys.executable} -m tests.test_sublayer_selfattention


## Feed-Forward Layer

Complete `FeedForwardLayer` in `hw4lib/model/sublayers.py`.


In [ ]:
!{sys.executable} -m tests.test_sublayer_feedforward


## Self-Attention Decoder Layer

Complete `SelfAttentionDecoderLayer`, including cache threading.


In [ ]:
!{sys.executable} -m tests.test_decoderlayer_selfattention


## Decoder-Only Transformer

Complete `DecoderOnlyTransformer`, including its per-layer cache.


In [ ]:
!{sys.executable} -m tests.test_transformer_decoder_only


## Greedy Decoding

Complete `generate_greedy` in `hw4lib/decoding/sequence_generator.py`.


In [ ]:
!{sys.executable} -m tests.test_decoding_greedy


## Sampling

Complete `_filter_logits` and `generate_sample` in
`hw4lib/decoding/sequence_generator.py`.


In [ ]:
!{sys.executable} -m tests.test_decoding_sampling


#### Optional: Beam Search

You may implement `generate_beam`. It is not required or graded.


In [ ]:
# Remove the leading # after implementing beam search.
# !{sys.executable} -m tests.test_decoding_beam


## Optimizer and Scheduler

The scheduler utility is provided. Run this check before testing trainer gradient
accumulation.


In [ ]:
!{sys.executable} -m tests.test_lr_scheduler_accumulation


## LM Trainer

Complete the TODOs in `hw4lib/trainers/lm_trainer.py`, including
gradient accumulation, validation, and generation.


In [ ]:
!{sys.executable} -m tests.test_lm_trainer_p1


## End-to-End Check

Run this after all tests above pass.


In [ ]:
!{sys.executable} -m tests.test_hw4p1_pipeline


# Experiments

Start with `mini_train` and `mini_val`. After the model trains correctly, update the
configuration for `train` and `val` and run your final experiment.


## Data

The setup section for your environment must set `HW4P1_DATA_ROOT` before you run
this cell.


In [ ]:
data_root_value = os.environ.get("HW4P1_DATA_ROOT")
if not data_root_value:
    raise RuntimeError("Complete the data setup for your environment first.")

DATA_ROOT = Path(data_root_value).expanduser().resolve()
for partition in ("mini_train", "mini_val", "train", "val", "test"):
    assert (DATA_ROOT / partition / "manifest.json").is_file(), partition

print("Data root:", DATA_ROOT)


## Config

The values below are only a starting point for the mini data. Choose your final
architecture and training settings, then change the partitions to `train`, `val`,
and `test` before final training.

For shorter experiments, set `data.max_train_windows_per_epoch` to a positive
integer (for example, `20000`). Each epoch draws that many distinct training
windows, capped at the partition size. `null` uses the whole training partition.
The seed and epoch determine the draw; windows may recur across epochs.
Validation and test still use every window in their selected partitions.

With a budget, an epoch is a sampled pass, not a full corpus pass. Compare runs
using windows/tokens processed as well as epochs. Cosine and warmup durations
use this loader's optimizer-update count, including gradient accumulation.
Rebuild the loaders and scheduler after changing the budget. For an existing
checkpoint, keep the original budget and scheduler settings when resuming.


In [ ]:
%%writefile config.yaml

Name: "hw4p1-mini"

###### Dataset -----------------------------------------------------------------
data:
  root            : ""           # Filled from HW4P1_DATA_ROOT in the next cell
  train_partition : "mini_train" # Change to train for final training
  val_partition   : "mini_val"   # Change to val for final training
  test_partition  : "mini_val"   # Change to test for final evaluation
  context_length  : 128
  stride          : 128
  batch_size      : 32
  max_train_windows_per_epoch: null # Positive integer for a shorter sampled epoch
  num_workers     : 2             # Set to 0 when debugging dataloaders

###### Network Specs -----------------------------------------------------------
model: # Decoder-Only Transformer (HW4P1)
  num_layers      : 2
  d_model         : 128
  num_heads       : 4             # Query heads
  num_kv_heads    : 2             # MHA: num_heads, GQA: 2+, MQA: 1
  d_ff            : 512
  dropout         : 0.1
  layer_drop_rate : 0.0
  weight_tying    : True
  use_rope        : True
  rope_base       : 10000.0

###### Training ----------------------------------------------------------------
training:
  epochs                          : 2
  gradient_accumulation_steps     : 1
  seed                            : 11785
  label_smoothing                 : 0.0
  use_wandb                       : False  # Set True to send run logs to W&B
  wandb_project                   : "hw4p1"
  wandb_entity                    : ""     # W&B username or team; blank uses default
  wandb_run_id                    : "none" # Set this only when resuming a W&B run
  wandb_log_checkpoints           : False  # Large files; local checkpoints are always saved

###### Optimizer ---------------------------------------------------------------
optimizer:
  name         : "adamw"          # sgd, adam, adamw
  lr           : 0.0005
  weight_decay : 0.01

  param_groups: []

  layer_decay:
    enabled    : False
    decay_rate : 0.75

  sgd:
    momentum   : 0.9
    nesterov   : True
    dampening  : 0

  adam:
    betas      : [0.9, 0.999]
    eps        : 1.0e-8
    amsgrad    : False

  adamw:
    betas      : [0.9, 0.95]
    eps        : 1.0e-8
    amsgrad    : False

###### Scheduler ---------------------------------------------------------------
scheduler:
  name: "cosine"                  # reduce_lr, cosine, cosine_warm

  reduce_lr:
    mode           : "min"
    factor         : 0.1
    patience       : 2
    threshold      : 0.0001
    threshold_mode : "rel"
    cooldown       : 0
    min_lr         : 1.0e-7
    eps            : 1.0e-8

  cosine:
    T_max      : 2                 # Match training.epochs
    eta_min    : 1.0e-5
    last_epoch : -1

  cosine_warm:
    T_0        : 2
    T_mult     : 2
    eta_min    : 1.0e-5
    last_epoch : -1

  warmup:
    enabled      : False
    type         : "linear"
    epochs       : 1
    start_factor : 0.1
    end_factor   : 1.0


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

config["data"]["root"] = str(DATA_ROOT)
seed = int(config["training"]["seed"])
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

CONFIG_PATH.write_text(
    yaml.safe_dump(config, sort_keys=False),
    encoding="utf-8",
)
print(CONFIG_PATH.read_text(encoding="utf-8"))


## Weights & Biases

W&B is controlled by `training.use_wandb` in `config.yaml`. Store a secret named
`WANDB_API_KEY` in Colab or Kaggle, or export it in your local/PSC environment.
If no secret is found, the next cell uses a hidden prompt. Never paste the key into
the notebook source.


In [ ]:
import getpass

if config["training"].get("use_wandb", False):
    try:
        import wandb
    except ImportError as exc:
        raise RuntimeError(
            "Install the requirements before enabling W&B."
        ) from exc

    wandb_key = os.environ.get("WANDB_API_KEY", "")

    if not wandb_key:
        try:
            from google.colab import userdata
            wandb_key = userdata.get("WANDB_API_KEY") or ""
        except Exception:
            pass

    if not wandb_key:
        try:
            from kaggle_secrets import UserSecretsClient
            wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY") or ""
        except Exception:
            pass

    if not wandb_key:
        wandb_key = getpass.getpass("W&B API key: ")
    if not wandb_key:
        raise RuntimeError("WANDB_API_KEY is required when use_wandb is True.")

    os.environ["WANDB_API_KEY"] = wandb_key
    del wandb_key
    wandb.login()
    print("W&B project:", config["training"]["wandb_project"])
else:
    print("W&B logging is disabled in config.yaml.")


## Tokenizer and Datasets


In [ ]:
tokenizer = GPT2Tokenizer()
data_config = config["data"]

train_dataset = LMDataset(
    data_config["root"],
    data_config["train_partition"],
    context_length=data_config["context_length"],
    stride=data_config["stride"],
)
val_dataset = LMDataset(
    data_config["root"],
    data_config["val_partition"],
    context_length=data_config["context_length"],
)
test_dataset = LMDataset(
    data_config["root"],
    data_config["test_partition"],
    context_length=data_config["context_length"],
)

print("Vocabulary size:", tokenizer.vocab_size)
print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Test examples:", len(test_dataset))


## Dataloaders


In [ ]:
loader_options = {
    "batch_size": data_config["batch_size"],
    "num_workers": data_config["num_workers"],
    "pin_memory": device == "cuda",
}
train_sampler = EpochWindowSampler(
    train_dataset,
    max_windows=data_config.get("max_train_windows_per_epoch"),
    seed=config["training"]["seed"],
)
train_loader = DataLoader(
    train_dataset,
    sampler=train_sampler,
    **loader_options,
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

verify_dataloader(train_loader)
inputs, targets = next(iter(train_loader))
print("Batch shape:", inputs.shape, targets.shape)

print(f"Training windows per epoch: {len(train_sampler):,} / {len(train_dataset):,}")
print(f"Training target tokens per epoch: {len(train_sampler) * data_config['context_length']:,}")
print("Batches (train / val / test):", len(train_loader), len(val_loader), len(test_loader))


## Model


In [ ]:
model = DecoderOnlyTransformer(
    **config["model"],
    max_len=config["data"]["context_length"],
    num_classes=tokenizer.vocab_size,
).to(device)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print("Unique parameters:", f"{parameter_count:,}")


## Trainer

The trainer logs losses, perplexities, learning rate, attention plots, and generated
samples to W&B when `use_wandb` is `True`.


In [ ]:
os.environ.setdefault(
    "HW4P1_OUTPUT_ROOT",
    str(PROJECT_ROOT / "expts"),
)

trainer = LMTrainer(
    model=model,
    tokenizer=tokenizer,
    config=config,
    run_name=config["Name"],
    config_file=str(CONFIG_PATH),
    device=device,
)
trainer.optimizer = create_optimizer(model, config["optimizer"])
trainer.scheduler = create_scheduler(
    trainer.optimizer,
    config["scheduler"],
    train_loader,
    config["training"]["gradient_accumulation_steps"],
)


### Resume from a Checkpoint

Skip this cell for a new run. For a resumed run, replace the path below and run it
before training.


In [ ]:
# trainer.load_checkpoint(str(trainer.last_model_path))


# Train

Run the mini configuration first. For final training, update the config, rebuild the
datasets, model, and trainer, and run this cell again.


In [ ]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=config["training"]["epochs"],
)


# Evaluate

Run this section after final training and model selection. Load your best checkpoint
before evaluating the test partition.


In [ ]:
trainer.load_checkpoint(str(trainer.best_model_path))
test_metrics, test_generated_results = trainer.evaluate(test_loader)
print(test_metrics)


# Evaluation Check

Confirm that final evaluation completed before generating submission evidence.


In [ ]:
required_metrics = {"ce_loss_token", "ce_loss_char", "perplexity_token", "perplexity_char"}
missing_metrics = required_metrics.difference(test_metrics)
if missing_metrics:
    raise RuntimeError(f"Evaluation is missing metrics: {sorted(missing_metrics)}")
if not all(np.isfinite(float(test_metrics[name])) for name in required_metrics):
    raise RuntimeError("Evaluation metrics must be finite.")
print("Evaluation metrics are ready for submission evidence.")


# Submission Evidence — DO NOT MODIFY

Run this cell once after final evaluation. It creates the required evidence files and
`handin.tar.gz` from the selected checkpoint, trainer history, configuration, test
metrics, and generation results. Do not edit this cell or the generated files.

If W&B is enabled, take a screenshot of the run page and save it in the repository
root as `training_screenshot.png`. It must show the run name or ID and the complete
training/validation PPL plot. Do not include an API key. The notebook always generates
`training_curve.png` from local trainer history.

This marker does not make a notebook cell tamper-proof. Gradescope will validate the
training results, run metadata, and submitted code independently.


In [ ]:
# ======================== DO NOT MODIFY THIS CELL ========================
from datetime import datetime, timezone
import json
import platform
import shutil
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import torch

EVIDENCE_SCHEMA_VERSION = 1
EVIDENCE_DIR = PROJECT_ROOT / "submission_evidence"
EVIDENCE_DIR.mkdir(exist_ok=True)


def _json_ready(value):
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().item() if value.numel() == 1 else value.detach().cpu().tolist()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if hasattr(value, "item"):
        return value.item()
    return str(value)


def _write_json(filename, payload):
    path = EVIDENCE_DIR / filename
    path.write_text(
        json.dumps(_json_ready(payload), indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    return path


selected_checkpoint = Path(trainer.best_model_path)
if not selected_checkpoint.is_file():
    raise FileNotFoundError(
        "Best checkpoint is missing. Complete training before generating evidence."
    )

screenshot_source = PROJECT_ROOT / "training_screenshot.png"
wandb_enabled = bool(config["training"].get("use_wandb", False))
if wandb_enabled and not screenshot_source.is_file():
    raise FileNotFoundError(
        "W&B is enabled. Save its run page as training_screenshot.png first."
    )
if screenshot_source.is_file():
    with screenshot_source.open("rb") as screenshot_file:
        if screenshot_file.read(8) != b"\x89PNG\r\n\x1a\n":
            raise ValueError("training_screenshot.png must be a valid PNG file.")
    shutil.copy2(screenshot_source, EVIDENCE_DIR / "training_screenshot.png")

history = _json_ready(trainer.training_history)
validation_records = [
    record for record in history
    if isinstance(record.get("val"), dict)
    and "ce_loss_char" in record["val"]
]
if not validation_records:
    raise RuntimeError("No validation metrics were found in trainer.training_history.")
selected_record = min(
    validation_records,
    key=lambda record: record["val"]["ce_loss_char"],
)

partition_manifests = {}
for partition in sorted({
    config["data"]["train_partition"],
    config["data"]["val_partition"],
    config["data"]["test_partition"],
}):
    manifest_path = DATA_ROOT / partition / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(manifest_path)
    partition_manifests[partition] = {
        "path": str(manifest_path.relative_to(DATA_ROOT)),
    }

dataset_info_path = DATA_ROOT / "dataset_info.json"
dataset_info = (
    {"path": dataset_info_path.name}
    if dataset_info_path.is_file()
    else None
)
config_file = {"filename": CONFIG_PATH.name}
generated_utc = datetime.now(timezone.utc).isoformat()

selection_metrics_path = _write_json(
    "selection_metrics.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "selection_rule": "minimum validation ce_loss_char",
        "selected_epoch": selected_record["epoch"],
        "validation_metrics": selected_record["val"],
        "checkpoint_filename": selected_checkpoint.name,
    },
)
test_metrics_path = _write_json(
    "test_metrics.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "checkpoint_filename": selected_checkpoint.name,
        "metrics": test_metrics,
    },
)
generated_samples_path = _write_json(
    "generated_samples.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "checkpoint_filename": selected_checkpoint.name,
        "samples_by_strategy": test_generated_results,
    },
)
training_summary_path = _write_json(
    "training_summary.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "run_name": config["Name"],
        "generated_utc": generated_utc,
        "seed": config["training"]["seed"],
        "epochs_completed": trainer.current_epoch,
        "global_optimizer_steps": trainer.global_optimizer_step,
        "unique_parameters": parameter_count,
        "device": str(device),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "config_file": config_file,
        "dataset_info": dataset_info,
        "partition_manifests": partition_manifests,
        "training_history": history,
    },
)
checkpoint_manifest_path = _write_json(
    "checkpoint_manifest.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "filename": selected_checkpoint.name,
        "selected_epoch": selected_record["epoch"],
        "global_optimizer_steps": trainer.global_optimizer_step,
    },
)

wandb_run = getattr(trainer, "wandb_run", None)
wandb_run_path = _write_json(
    "wandb_run.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "enabled": bool(config["training"].get("use_wandb", False)),
        "entity": getattr(wandb_run, "entity", None),
        "project": getattr(wandb_run, "project", config["training"].get("wandb_project")),
        "run_id": getattr(wandb_run, "id", None),
        "run_name": getattr(wandb_run, "name", config["Name"]),
        "url": getattr(wandb_run, "url", None),
    },
)

evidence_plot_path = EVIDENCE_DIR / "training_curve.png"
figure, axis = plt.subplots(figsize=(9, 5))
for split in ("train", "val", "test"):
    points = [
        (record["epoch"], record[split]["perplexity_char"])
        for record in history
        if isinstance(record.get(split), dict)
        and "perplexity_char" in record[split]
    ]
    if points:
        axis.plot(
            [point[0] for point in points],
            [point[1] for point in points],
            marker="o",
            label=split,
        )
axis.set_title(
    f"{config['Name']} | {parameter_count:,} parameters | "
    f"{trainer.global_optimizer_step:,} optimizer steps"
)
axis.set_xlabel("Epoch")
axis.set_ylabel("Character perplexity")
axis.grid(alpha=0.3)
axis.legend()
figure.tight_layout()
figure.savefig(evidence_plot_path, dpi=160)
plt.show()
plt.close(figure)

shutil.copy2(CONFIG_PATH, EVIDENCE_DIR / "config.yaml")
shutil.copy2(trainer.expt_root / "model_arch.txt", EVIDENCE_DIR / "model_arch.txt")

evidence_files = sorted(
    path for path in EVIDENCE_DIR.iterdir()
    if path.is_file() and path.name != "submission_manifest.json"
)
submission_manifest_path = _write_json(
    "submission_manifest.json",
    {
        "schema_version": EVIDENCE_SCHEMA_VERSION,
        "generated_utc": generated_utc,
        "files": {
            path.name: {}
            for path in evidence_files
        },
    },
)

archive_path = PROJECT_ROOT / "handin.tar.gz"
with tarfile.open(archive_path, "w:gz") as archive:
    for source_dir in (PROJECT_ROOT / "mytorch", PROJECT_ROOT / "hw4lib"):
        archive.add(
            source_dir,
            arcname=f"handin/{source_dir.name}",
            filter=lambda info: None if "__pycache__" in info.name else info,
        )
    for artifact_path in sorted(EVIDENCE_DIR.iterdir()):
        if artifact_path.is_file():
            archive.add(
                artifact_path,
                arcname=f"handin/{artifact_path.name}",
            )

trainer.cleanup()
print("Evidence directory:", EVIDENCE_DIR)
print("Submission archive:", archive_path)
print("Checkpoint weights were not added to the archive.")
# ====================== END DO NOT MODIFY CELL ======================


# Final Submission Check

Run this after `handin.tar.gz` has been created. It reruns the required code tests and validates the training evidence.


In [ ]:
!{sys.executable} -m tests.test_hw4p1
